In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LassoCV, LinearRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.base import TransformerMixin, BaseEstimator, clone
from IPython.display import Markdown
import seaborn as sns
import warnings
warnings.simplefilter('ignore')
np.random.seed(1234)

In [ ]:
#from google.colab import files
#uploaded = files.upload()

In [ ]:
data = pd.read_stata('/home/onyxia/work/data/dataPrivatePublic.dta')

## Nettoyage de la base

In [ ]:
# ----------------------------
# Helpers
# ----------------------------
def is_str(x):
    return isinstance(x, str)

def to_str(s: pd.Series) -> pd.Series:
    """Safely cast to string while preserving NaN."""
    return s.astype("string")

def inrange(s: pd.Series, a: int, b: int) -> pd.Series:
    return s.between(a, b, inclusive="both")

# Cast key columns to string where Stata compares to "..."
for col in ["nregion", "motins", "exper", "rsqstat", "temps", "zus", "salaire", "cemploi",
            "sexe", "nation", "lot", "ale"]:
    if col in data.columns:
        data[col] = to_str(data[col])

# Make sure numeric columns are numeric where needed
for col in ["ndem", "CS", "nenf", "age", "mois_saisie_occ"]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

Ici, je recrée toutes les dummys comme dans le code stata pour les variables X.

In [ ]:
df = data.copy()

# ----------------------------
# 1) Region dummies
# ----------------------------
df["IdF"] = (df["nregion"] == "116").astype(int)
df["North"] = (df["nregion"] == "311").astype(int)

# ----------------------------
# 2) Unemployment reason (motins)
# ----------------------------
df["EconLayoff"]   = (df["motins"] == "1").astype(int)
df["PersLayoff"]   = (df["motins"] == "2").astype(int)
df["EndCDD"]       = (df["motins"] == "4").astype(int)
df["EndInterim"]   = (df["motins"] == "5").astype(int)
df["Otherend"]     = (1 - df["EconLayoff"] - df["PersLayoff"] - df["EndCDD"] - df["EndInterim"]).astype(int)

# ----------------------------
# 3) Experience in job (exper)
# ----------------------------
df["exper0"] = (df["exper"] == "00").astype(int)
df["exper1_5"] = df["exper"].isin(["01","02","03","04","05"]).astype(int)
df["experM5"] = (1 - df["exper0"] - df["exper1_5"]).astype(int)

# ----------------------------
# 4) Statistical risk (rsqstat; this is a risk computed by the ANPE of an individual remaining in unemployment for a long term)
# ----------------------------
df["rsqstat2"] = (df["rsqstat"] == "RS2").astype(int)
df["rsqstat3"] = (df["rsqstat"] == "RS3").astype(int)
df["Orsqstat"] = (1 - df["rsqstat2"] - df["rsqstat3"]).astype(int)

# ----------------------------
# 5) Full-time search (temps)
# ----------------------------
df["tempcomp"] = (df["temps"] == "1").astype(int)
df["Otemp"] = (1 - df["tempcomp"]).astype(int)

# ----------------------------
# 6) Sensitive suburban area (zus)
# ----------------------------
df["dezus"] = (df["zus"] == "ZU").astype(int)

# ----------------------------
# 7) Wage target (salaire)
# ----------------------------
df["salaireA"] = (df["salaire"] == "A").astype(int)
df["salaireB"] = (df["salaire"] == "B").astype(int)
df["salaireC"] = (df["salaire"] == "C").astype(int)
df["salaireD"] = (df["salaire"] == "D").astype(int)
df["salaireE"] = (df["salaire"] == "E").astype(int)
# Stata: (salaire=="G") + (salaire=="")  -> "No wage target" includes missing-as-empty
df["salaireG"] = ((df["salaire"] == "G") | (df["salaire"].fillna("") == "")).astype(int)

# ----------------------------
# 8) Employment component (cemploi)
# ----------------------------
df["ce1"] = (df["cemploi"] == "CE1").astype(int)
df["ce2"] = (df["cemploi"] == "CE2").astype(int)
df["cemiss"] = (df["cemploi"].fillna("") == "").astype(int)

# ----------------------------
# 9) First unemployment spell (ndem==1)
# ----------------------------
df["primo"] = (df["ndem"] == 1).astype(int)

# ----------------------------
# 10) Occupation categories (CS)
# ----------------------------
df["Cadre"]    = (df["CS"] == 3).astype(int)
df["Techn"]    = (df["CS"] == 4).astype(int)
df["EmployQ"]  = (df["CS"] == 51).astype(int)
df["EmployNQ"] = (df["CS"] == 56).astype(int)
df["OuvrQ"]    = (df["CS"] == 61).astype(int)
df["OuvrNQ"]   = df["CS"].isin([66, 99]).astype(int)

# ----------------------------
# 11) Nationality groups (nation is string in Stata code)
# IMPORTANT: Stata uses string comparisons like nation<"31".
# We mimic that by keeping as string and using lexicographic comparison.
# If your nation codes are numeric, convert to zero-padded strings first.
# ----------------------------
nation = df["nation"].fillna("")

df["African"] = ((nation >= "31") & (nation <= "49")).astype(int)

df["EasternEurope"] = (
    ((nation >= "90") & (nation <= "98")) |
    nation.isin(["24", "25", "27"])
).astype(int)

df["SouthEuropTurkey"] = nation.isin(
    ["02","03","14","19","21","22","24","27","26"]
).astype(int)

# The original Stata later does: gen French=etranger (but etranger is not created here).
# If you already have a column `etranger` coded 0/1, use it; otherwise comment this out.
if "etranger" in df.columns:
    df["French"] = pd.to_numeric(df["etranger"], errors="coerce").fillna(0).astype(int)
else:
    # placeholder: set to NaN if not available
    df["French"] = np.nan

df["Otherregion"] = (1 - df["IdF"] - df["North"]).astype(int)

# Stata had: Othernation=1-French-African /*-EasternEurope-SouthEuropTurkey*/
# (they commented out subtracting EasternEurope and SouthEuropTurkey)
if df["French"].notna().any():
    df["Othernation"] = (1 - df["French"].fillna(0).astype(int) - df["African"]).astype(int)
else:
    df["Othernation"] = np.nan

# ----------------------------
# 12) Children and sex
# ----------------------------
df["nochild"] = (df["nenf"] == 0).astype(int)
df["onechild"] = (df["nenf"] == 1).astype(int)
df["twoormorechild"] = (df["nenf"] > 1).astype(int)

df["woman"] = (df["sexe"] == "2").astype(int)

# ----------------------------
# 13) Type of operator (lot -> TypeOPP -> dummies)
# ----------------------------
df["TypeOPP"] = ""

counseling_lots = {"6","10","14","15","16","17"}
interim_lots    = {"12","13","19","24","25"}
insertion_lots  = {"7","18","22","23"}

df.loc[df["lot"].isin(counseling_lots), "TypeOPP"] = "Counseling"
df.loc[df["lot"].isin(interim_lots),    "TypeOPP"] = "Interim"
df.loc[df["lot"].isin(insertion_lots),  "TypeOPP"] = "Insertion"

df["conseil"]   = (df["TypeOPP"] == "Counseling").astype(int)
df["interim"]   = (df["TypeOPP"] == "Interim").astype(int)
df["insertion"] = (df["TypeOPP"] == "Insertion").astype(int)

# ----------------------------
# 14) Cleaned local area code (ale -> alec)
# ----------------------------
df["alec"] = df["ale"]

recode_alec = {
    "77111": "77103",
    "75884": "75861",
    "59121": "59113",
    "42002": "42024",
    "42040": "42024",
    "26031": "26023",
}
df["alec"] = df["alec"].replace(recode_alec)

# ----------------------------
# 15) Dominant operator type by area (alec)
# Replicates the Stata loop that computes means by area and assigns them back.
# ----------------------------
area_means = (
    df.groupby("alec")[["interim", "insertion", "conseil"]]
      .mean()
      .rename(columns={"interim":"Einterim","insertion":"Einsertion","conseil":"Econseil"})
)

df = df.join(area_means, on="alec")

df["Interim"] = ((df["Einterim"] > df["Econseil"]) & (df["Einterim"] > df["Einsertion"])).astype(int)
df["Insertion"] = ((df["Einsertion"] > df["Econseil"]) & (df["Einsertion"] > df["Einterim"])).astype(int)
df["Conseil"] = ((df["Econseil"] > df["Einterim"]) & (df["Econseil"] > df["Einsertion"])).astype(int)

# "nc" copies
df["Interimnc"] = df["Interim"]
df["Insertionnc"] = df["Insertion"]
df["Conseilnc"] = df["Conseil"]

# AreaTypeOPP label
df["AreaTypeOPP"] = np.where(df["Interim"] == 1, "Interim",
                      np.where(df["Insertion"] == 1, "Insertion",
                      np.where(df["Conseil"] == 1, "Counseling", "")))

# ----------------------------
# 16) Cohort of assignment (quarters)
# NOTE: The Stata code has a bug for Q4 (it repeats 7-9).
# Here we implement the intended quarters: Q4 = 10-12.
# ----------------------------
df["Q1"] = inrange(df["mois_saisie_occ"], 1, 3).astype(int)
df["Q2"] = inrange(df["mois_saisie_occ"], 4, 6).astype(int)
df["Q3"] = inrange(df["mois_saisie_occ"], 7, 9).astype(int)
df["Q4"] = inrange(df["mois_saisie_occ"], 10, 12).astype(int)


In [ ]:

# *******************************************************
# * Outcome variables for Cost benefit analysis
# *******************************************************

# --- Cost variables ---
df["cost_opp"] = df["acceptationOPP"] * (0.3 + df["EMPLOI_AR110_6MOIS"] * 0.35 + 0.35 * df["SUCCES_OPP_6MOIS"]) * 3000
df["cost_cve"] = df["acceptationCVE"] * 657
df["cost_cla"] = (1 - df["acceptationOPP"] - df["acceptationCVE"]) * 120
df["cost"] = df["cost_opp"] + df["cost_cve"] + df["cost_cla"]

# --- Estimation of monthly UB (UI) ---
pi = 0.5
df["estimated_monthly_ui"] = 0.0

# wage initialisé à 0 puis rempli selon la catégorie salaire
df["wage"] = 0.0

df.loc[df["salaire"].eq("A"), "wage"] = pi * 1220 + (1 - pi) * 1349
df.loc[df["salaire"].eq("B"), "wage"] = pi * 1350 + (1 - pi) * 1549
df.loc[df["salaire"].eq("C"), "wage"] = pi * 1550 + (1 - pi) * 1799
df.loc[df["salaire"].eq("D"), "wage"] = pi * 1800 + (1 - pi) * 2200
df.loc[df["salaire"].eq("E"), "wage"] = 2600

# salaire == "G" = manquant (Stata: .)
df.loc[df["salaire"].eq("G"), "wage"] = np.nan


# *******************************************************
# * Imputation wage when missing via ordered logit
# *******************************************************

# catsal: 1..5 selon A..E ; 0 sinon
df["catsal"] = 0
df.loc[df["salaire"].eq("A"), "catsal"] = 1
df.loc[df["salaire"].eq("B"), "catsal"] = 2
df.loc[df["salaire"].eq("C"), "catsal"] = 3
df.loc[df["salaire"].eq("D"), "catsal"] = 4
df.loc[df["salaire"].eq("E"), "catsal"] = 5

Xw = [
    "nivetude1","nivetude3","nivetude4","Cadre","Techn","EmployQ","EmployNQ","OuvrQ",
    "agegr2635","agegr3645","agegr4655","agegr56","woman","marie","onechild",
    "twoormorechild","French","African","EasternEurope","SouthEuropTurkey","IdF","North",
    "ce1","ce2","EconLayoff","PersLayoff","EndCDD","EndInterim","exper0","exper1_5",
    "rsqstat2","rsqstat3","tempcomp","dezus","primo"
]

# Estimation sur ceux qui n'ont PAS salaire == "G"
mask_obs = ~df["salaire"].eq("G")
df_obs = df.loc[mask_obs, ["catsal"] + Xw].dropna()

# Ordered logit (équivalent Stata: ologit catsal Xw)
y = df_obs["catsal"]
X = df_obs[Xw]

# statsmodels gère l'intercept via "has_constant" ou en ajoutant un constant selon le modèle;
# OrderedModel n'utilise pas sm.add_constant, il gère les seuils (cutpoints) lui-même.
model = OrderedModel(y, X, distr="logit")
res = model.fit(method="bfgs", disp=False)

# Probabilités prédites pour toutes les lignes (on met NaN si covariates manquantes)
X_all = df[Xw]
pred_probs = pd.DataFrame(res.model.predict(res.params, exog=X_all), index=df.index)

# Les colonnes correspondent aux catégories ordonnées (1..5)
pred_probs.columns = [1, 2, 3, 4, 5]

# Noms Stata PA..PE
df["PA"] = pred_probs[1]
df["PB"] = pred_probs[2]
df["PC"] = pred_probs[3]
df["PD"] = pred_probs[4]
df["PE"] = pred_probs[5]

# Imputation wage pour salaire == "G"
wA = pi * 1220 + (1 - pi) * 1349
wB = pi * 1350 + (1 - pi) * 1549
wC = pi * 1550 + (1 - pi) * 1799
wD = pi * 1800 + (1 - pi) * 2200
wE = 2600

mask_missing = df["salaire"].eq("G")
df.loc[mask_missing, "wage"] = (
    df.loc[mask_missing, "PA"] * wA +
    df.loc[mask_missing, "PB"] * wB +
    df.loc[mask_missing, "PC"] * wC +
    df.loc[mask_missing, "PD"] * wD +
    df.loc[mask_missing, "PE"] * wE
)

# --- Estimated UI using replacement ratio and reduced activity ---
df["Replacement"] = 0.72
df.loc[df["wage"] > 1300, "Replacement"] = 0.68
df.loc[df["wage"] > 1900, "Replacement"] = 0.64

df["estimated_monthly_ui"] = df["Replacement"] * df["wage"]
df["estimated_ui"] = df["estimated_monthly_ui"] * df["duree_listes_horsAR"] / 365 * 31
df["Total_expenses"] = df["estimated_ui"] + df["cost"]


# *******************************************************
# * Some treatment variables
# *******************************************************
df["acceptationCLA_6MOIS"] = 1 - df["acceptationCVE_6MOIS"] - df["acceptationOPP_6MOIS"]

df["acceptation"] = np.nan
df.loc[df["OPP"] == 1, "acceptation"] = df.loc[df["OPP"] == 1, "acceptationOPP_6MOIS"]
df.loc[df["CVE"] == 1, "acceptation"] = df.loc[df["CVE"] == 1, "acceptationCVE_6MOIS"]
df.loc[(df["OPP"] == 0) & (df["CVE"] == 0), "acceptation"] = 0


## Partie DML

Dans cette partie, nous essayons de retrouver les résulats du LATE en effectuant un DML.










Commençons par des régressions naïves.

In [ ]:
#Nous commençons par travailler sur un sous-échantillon de df
sample_cveopp = df.loc[df["SAMPLE_CVEOPP"] == 1].copy()

In [ ]:
#Les variables de contrôle
X_limited = ["North", "IdF", "salaireG", "duree_listes_horsAR", "Insertion", "Q2", "agegr56", "exper0", "agegr4655", "French", "agegr3645", "Interim", "ce2", "Q3", "EndInterim", "salaireB", "primo", "agegr2635", "nivetude4", "tempcomp", "nivetude3", "salaireD", "EconLayoff", "PersLayoff", "salaireC", "Q1", "African", "salaireE"]

#Le traitement, ici le traitement effectif
d1 = "acceptationCVE_6MOIS"
d2 = "acceptationOPP_6MOIS"

#L'outcome
y="EMPLOI_6MOIS"

Faisons une première régression naïve de y sur d1, d2.

In [ ]:
# Baseline OLS
w = sample_cveopp["POIDS_PZ_6MOIS"].astype(float)
X = sm.add_constant(sample_cveopp[["acceptationCVE_6MOIS", "acceptationOPP_6MOIS"]])
y = sample_cveopp["EMPLOI_6MOIS"]

lm0 = sm.WLS(y, X, weights=w).fit(cov_type="HC1")
vc0 = lm0.cov_params()
print("b_CVE =", lm0.params["acceptationCVE_6MOIS"], "se =", lm0.bse["acceptationCVE_6MOIS"])
print("b_OPP =", lm0.params["acceptationOPP_6MOIS"], "se =", lm0.bse["acceptationOPP_6MOIS"])


Now, I estimate with a set of controls (which are that that play the most in term of imbalance between the treated and the non-treated).

In [ ]:
# Regression on baseline controls
X_limited_corrected = [
    "North", "femme", "IdF", "salaireG", "Insertion", "Q2",
     "exper", "French", "age", "Interim", "ce2", "Q3", "EndInterim", "salaireB", "primo", "nivetude4",
    "tempcomp", "nivetude3", "salaireD", "EconLayoff", "PersLayoff",
    "salaireC", "Q1", "African", "salaireE", "duree_listes_horsAR", "marie", "nenf", "ce1",
]

# Ensure 'exper' is numeric before proceeding to prevent ValueError from statsmodels
sample_cveopp["exper"] = pd.to_numeric(sample_cveopp["exper"], errors="coerce")

varlist = [d1] + [d2] + X_limited_corrected
w = sample_cveopp["POIDS_PZ_6MOIS"].astype(float)

# Explicitly convert the DataFrame subset to float to prevent any object dtypes
X_df = sample_cveopp[varlist].astype(float)
X = sm.add_constant(X_df)
y = sample_cveopp["EMPLOI_6MOIS"]

lmC = sm.WLS(y, X, weights=w).fit(cov_type="HC1")
vcC = lmC.cov_params()
print("b_CVE =", lmC.params["acceptationCVE_6MOIS"], "se =", lmC.bse["acceptationCVE_6MOIS"])
print("b_OPP =", lmC.params["acceptationOPP_6MOIS"], "se =", lmC.bse["acceptationOPP_6MOIS"])

## First stage of the DML

Ici, l'idée est d'essayer de voir si, en partant de l'hypothèse de unconfoundedness, nous parvenons à retrouver un résultat moyen causal qui soit du même ordre que le résultat du LATE de l'article (résultat sur les compliers seulement). Pour se faire, nous estimons les paramètres $\alpha_1$ et $\alpha_2$ du modèle statistique partiellement linéaire suivant,


\begin{equation}
 Y = \alpha_1 \mathbb{1}(Z=1) +  \alpha_2 \mathbb{1}(Z=2) + + g(X) + \epsilon,
\end{equation}
 avec $$\quad E (\epsilon | \mathbb{1}(Z=1), \mathbb{1}(Z=2), X) = 0.$$

A traduire

For $\tilde Y = Y- E(Y|X)$ and $\tilde D= D- E(D|X)$, we can write
$$
\tilde Y = \alpha \tilde D + U, \quad E (U |\tilde D) =0.
$$
Parameter $\alpha$ is then estimated using cross-fitting approach to obtain the residuals $\tilde D$ and $\tilde Y$.
The algorithm comsumes $Y, D, X$, and machine learning methods for learning the residuals $\tilde Y$ and $\tilde D$, where
the residuals are obtained by cross-validation (cross-fitting).

The statistical parameter $\alpha$ has a causal interpretation of being the effect of $D$ on $Y$ in the causal DAG $$ D\to Y, \quad X\to (D,Y)$$ or the counterfactual outcome model with conditionally exogenous (conditionally random) assignment of treatment $D$ given $X$:
$$
Y(d) = d\alpha + g(X) + U(d),\quad  U(d) \text{ indep } D |X, \quad Y = Y(D), \quad U = U(D).
$$


### I define a transformer that constructs the engineered features for controls

In [ ]:
!pip install formulaic

In [ ]:
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LassoCV, LinearRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.base import TransformerMixin, BaseEstimator, clone
from formulaic import Formula
from sklearn.neural_network import MLPRegressor
from sklearn.neural_network import MLPClassifier

In [ ]:
X_limited_corrected = [
    "North", "femme", "IdF", "salaireG", "Insertion", "Q2",
     "exper", "French", "age", "Interim", "ce2", "Q3", "EndInterim", "salaireB", "primo", "nivetude4",
    "tempcomp", "nivetude3", "salaireD", "EconLayoff", "PersLayoff",
    "salaireC", "Q1", "African", "salaireE", "duree_listes_horsAR", "marie", "nenf", "ce1",
]
sample_cveopp["exper"] = pd.to_numeric(sample_cveopp["exper"], errors="coerce")
sample_cveopp[X_limited_corrected].describe()

In [ ]:
class FormulaTransformer(TransformerMixin, BaseEstimator):

    def __init__(self, formula, array=False):
        self.formula = formula
        self.array = array

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        df = Formula(self.formula).get_model_matrix(X)
        if self.array:
            return df.values
        return df

In [ ]:
transformer = FormulaTransformer(
    "0 "
    "+ poly(age, degree=3, raw=True)"
    "+ poly(exper, degree=3, raw=True)"
    "+ poly(duree_listes_horsAR, degree=3, raw=True)"
    "+ poly(nenf, degree=2, raw=True)"
    "+ North + IdF + French + African + femme + marie"
    "+ Interim + EndInterim + tempcomp"
    "+ ce1 + ce2"
    "+ nivetude3 + nivetude4"
    "+ salaireB + salaireC + salaireD + salaireE"
    "+ Q1 + Q2 + Q3"
    "+ EconLayoff + PersLayoff"
    "+ primo + Insertion"
    "+ age:exper"
    "+ femme:nenf"
    "+ femme:exper"
    "+ duree_listes_horsAR:exper"
    "+ French:IdF"
    "+ African:IdF"
    "+ nivetude3:exper"
    "+ nivetude4:exper",
    array=False
)

In [ ]:
transformer.fit_transform(sample_cveopp[X_limited_corrected]).describe()

### Estimating the ATE

In [ ]:
y = sample_cveopp["EMPLOI_6MOIS"].values
D1 = sample_cveopp['acceptationCVE_6MOIS'].values
D2 = sample_cveopp['acceptationOPP_6MOIS'].values
X = sample_cveopp[X_limited_corrected]

In [ ]:
def dml_two_treatments(
    X, D1, D2, y,
    modely, modeld1, modeld2=None,
    *, nfolds=5, classifier_y = False,
    classifier_d1=False, classifier_d2=False,
    clu=None, cluster=True,
    progress=True
):
    """
    DML for Partially Linear Model with TWO treatments (D1, D2) using cross-fitting.
    Stage 1 (cross-fitting):
    yhat = E[y|X]
    D1hat = E[D1|X]
    D2hat = E[D2|X]

    Residuals:
    resy = y - yhat
    resD1 = D1 - D1hat
    resD2 = D2 - D2hat

    Final stage: resy ~ 1 + resD1 + resD2
    Returns: point1, point2, stderr1, stderr2, yhat, D1hat, D2hat, resy, resD1, resD2, epsilon
    """

    if modeld2 is None:
        modeld2 = modeld1

    y  = np.asarray(y).ravel()
    D1 = np.asarray(D1).ravel()
    D2 = np.asarray(D2).ravel()

    cv = KFold(n_splits=nfolds, shuffle=True, random_state=123)

    # Required imports for tqdm and statsmodels.formula.api
    from tqdm.auto import tqdm
    import statsmodels.formula.api as smf

    pbar = tqdm(total=5, desc="DML steps", disable=not progress)

    # 1) yhat
    if classifier_y:
      yhat = cross_val_predict(
            modely, X, y, cv=cv, method="predict_proba", n_jobs=-1
        )[:, 1]
    else:
      yhat = cross_val_predict(modely, X, y, cv=cv, n_jobs=-1)
    pbar.update(1)

    # 2) D1hat
    if classifier_d1:
        D1hat = cross_val_predict(
            modeld1, X, D1, cv=cv, method="predict_proba", n_jobs=-1
        )[:, 1]
    else:
        D1hat = cross_val_predict(modeld1, X, D1, cv=cv, n_jobs=-1)
    pbar.update(1)

    # 3) D2hat
    if classifier_d2:
        D2hat = cross_val_predict(
            modeld2, X, D2, cv=cv, method="predict_proba", n_jobs=-1
        )[:, 1]
    else:
        D2hat = cross_val_predict(modeld2, X, D2, cv=cv, n_jobs=-1)
    pbar.update(1)

    # 4) Residuals
    resy  = y  - yhat
    resD1 = D1 - D1hat
    resD2 = D2 - D2hat

    dml_data = pd.DataFrame({
        "resy":  resy,
        "resD1": resD1,
        "resD2": resD2
    })
    pbar.update(1)

    # 5) Final OLS
    if cluster:
        if clu is None:
            raise ValueError("cluster=True but clu is None. Provide cluster ids in clu.")
        dml_data["clu"] = np.asarray(clu)

        ols_mod = smf.ols("resy ~ 1 + resD1 + resD2", data=dml_data).fit(
            cov_type="cluster", cov_kwds={"groups": dml_data["clu"]}
        )
    else:
        ols_mod = smf.ols("resy ~ 1 + resD1 + resD2", data=dml_data).fit()
    pbar.update(1)
    pbar.close()

    point1 = ols_mod.params["resD1"]
    point2 = ols_mod.params["resD2"]
    stderr1 = ols_mod.bse["resD1"]
    stderr2 = ols_mod.bse["resD2"]
    epsilon = ols_mod.resid

    return point1, point2, stderr1, stderr2, yhat, D1hat, D2hat, resy, resD1, resD2, epsilon

In [ ]:
def summary_two_treatments(
    point1, point2, stderr1, stderr2,
    yhat, D1hat, D2hat, resy, resD1, resD2, epsilon,
    X, D1, D2, y,
    *, name1="D1", name2="D2", binary_y=None,
    binary_d1=None, binary_d2=None
):
    """
    Summary function for DML with two treatments.

    Returns a DataFrame with one row per treatment.
    """

    # Convert to 1d arrays
    y = np.asarray(y).ravel()
    D1 = np.asarray(D1).ravel()
    D2 = np.asarray(D2).ravel()

    # If not specified, infer whether treatments are binary
    if binary_d1 is None:
        unique_d1 = np.unique(D1[~pd.isna(D1)])
        binary_d1 = len(unique_d1) <= 2 and set(unique_d1).issubset({0, 1})

    if binary_d2 is None:
        unique_d2 = np.unique(D2[~pd.isna(D2)])
        binary_d2 = len(unique_d2) <= 2 and set(unique_d2).issubset({0, 1})

    if binary_y is None:
        unique_y = np.unique(y[~pd.isna(y)])
        binary_y = len(unique_y) <= 2 and set(unique_y).issubset({0, 1})

    # Common metrics
    rmse_y = np.sqrt(np.mean(resy**2))
    rmse_eps = np.sqrt(np.mean(epsilon**2))

    # Treatment-specific metrics
    rmse_d1 = np.sqrt(np.mean(resD1**2))
    rmse_d2 = np.sqrt(np.mean(resD2**2))

    acc_d1 = np.mean(np.abs(resD1) < 0.5) if binary_d1 else np.nan
    acc_d2 = np.mean(np.abs(resD2) < 0.5) if binary_d2 else np.nan
    acc_y = np.mean(np.abs(resy) < 0.5) if binary_y else np.nan
    return pd.DataFrame({
        "estimate":   [point1, point2],
        "stderr":     [stderr1, stderr2],
        "lower":      [point1 - 1.96 * stderr1, point2 - 1.96 * stderr2],
        "upper":      [point1 + 1.96 * stderr1, point2 + 1.96 * stderr2],
        "rmse y":     [rmse_y, rmse_y],
        "accuracy y":[acc_y, acc_y],
        "rmse D":     [rmse_d1, rmse_d2],
        "accuracy D": [acc_d1, acc_d2],
        "rmse final": [rmse_eps, rmse_eps],
        "n":          [len(y), len(y)],
    }, index=[name1, name2])

### Learners of l(X) (E[Y|X]), m1(X) (E[D1|X]) and m2(X) E[D2|X]

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=123)

In [ ]:
learners_d = {

    "Logit": make_pipeline(transformer, StandardScaler(), LogisticRegressionCV(cv=cv, penalty='l2', solver='liblinear')),

    "RF": make_pipeline(transformer, RandomForestClassifier(n_estimators=100, min_samples_leaf=10, ccp_alpha=.001)),

    "Tree": make_pipeline(transformer, DecisionTreeClassifier(min_samples_leaf=10, ccp_alpha=.001)),

    "GBF": make_pipeline(transformer, GradientBoostingClassifier(max_depth=2, n_iter_no_change=5)),

    "MLP": make_pipeline(StandardScaler(),
                       MLPRegressor(hidden_layer_sizes=(50, 50, 50, 50),
                                    activation='relu',
                                    solver='adam',
                                    alpha=0.0001,
                                    batch_size=200,
                                    learning_rate='constant',
                                    learning_rate_init=0.001,
                                    max_iter=200,
                                    shuffle=True,
                                    random_state=None,
                                    tol=1e-4,
                                    verbose=False,
                                    warm_start=False,
                                    momentum=0.9,
                                    nesterovs_momentum=True,
                                    early_stopping=True,
                                    validation_fraction=0.2,
                                    beta_1=0.9,
                                    beta_2=0.999,
                                    epsilon=1e-08,
                                    n_iter_no_change=10)
                       )
}

In [ ]:
learners_y = {
    "Logit": make_pipeline(transformer, StandardScaler(), LogisticRegressionCV(cv=cv, penalty='l2', solver='liblinear')),

    "RF":  make_pipeline(transformer, RandomForestClassifier(n_estimators=100, min_samples_leaf=10, ccp_alpha=.001)),

    "Tree": make_pipeline(transformer, DecisionTreeClassifier(min_samples_leaf=10, ccp_alpha=.001)),

    "GBF": make_pipeline(transformer, GradientBoostingClassifier(max_depth=2, n_iter_no_change=5)),

    "MLP": make_pipeline(StandardScaler(),
                       MLPRegressor(hidden_layer_sizes=(50, 50, 50, 50),
                                    activation='relu',
                                    solver='adam',
                                    alpha=0.0001,
                                    batch_size=200,
                                    learning_rate='constant',
                                    learning_rate_init=0.001,
                                    max_iter=200,
                                    shuffle=True,
                                    random_state=None,
                                    tol=1e-4,
                                    verbose=False,
                                    warm_start=False,
                                    momentum=0.9,
                                    nesterovs_momentum=True,
                                    early_stopping=True,
                                    validation_fraction=0.2,
                                    beta_1=0.9,
                                    beta_2=0.999,
                                    epsilon=1e-08,
                                    n_iter_no_change=10)
                       )
}

### Estimate all combinations and compute goodness of fit

In [ ]:
results_summary = []
models = {}

for name_y, learner_y in learners_y.items():
    for name_d, learner_d in learners_d.items():

        print(f"Running combination: {name_y} / {name_d}")

        dml_out = dml_two_treatments(
            X=X,
            D1=D1,
            D2=D2,
            y=y,
            modely=learner_y,
            modeld1=learner_d,
            modeld2=learner_d,
            nfolds=5,
            classifier_y=True,
            classifier_d1=True,
            classifier_d2=True,
            cluster=False
        )

        (
            point1, point2, stderr1, stderr2,
            yhat, D1hat, D2hat,
            resy, resD1, resD2,
            epsilon
        ) = dml_out

        summ = summary_two_treatments(
            point1, point2, stderr1, stderr2,
            yhat, D1hat, D2hat, resy, resD1, resD2, epsilon,
            X, D1, D2, y,
            name1="CVE_treated", name2="OPP_treated"
        ).reset_index(names="treatment")

        summ["learner_y"] = name_y
        summ["learner_d"] = name_d

        results_summary.append(summ)

        models[f"{name_y}__{name_d}"] = {
            "summary": summ,
            "raw_output": dml_out
        }

results_summary = pd.concat(results_summary, ignore_index=True)

results_summary = results_summary.sort_values(
    by=["rmse y", "rmse D"],
    ascending=True
)


### Choose the best learner and compute the final estimation with the best combination

### Do the procedure for the variables EMPLOI_3MOIS, EMPLOI_9MOIS and EMPLOI_12MOIS to get the evolution of the causal effect

The results are really close to that to the LATE estimation ! This means that the DML procedure do work !

In [ ]:
#Code

### Display goodness of fit table

In [ ]:
print("\nGoodness of fit and DML results for all combinations:\n")
print(results_summary.to_string(index=False))

### Lasso

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=123)
lassoy = make_pipeline(transformer, StandardScaler(), LassoCV(cv=cv))
lassod1 = make_pipeline(transformer, StandardScaler(), LassoCV(cv=cv))
lassod2 = make_pipeline(transformer, StandardScaler(), LassoCV(cv=cv))
result_lasso = dml_two_treatments(X, D1,D2, y, lassoy, lassod1, lassod2, nfolds=5,cluster = False )

In [ ]:
table_lasso = summary_two_treatments(*result_lasso, X, D1,D2, y, name1="CVE_treated", name2="OPP_treated")
table_lasso

### Logistic regression

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=123)
logy = make_pipeline(transformer, StandardScaler(), LogisticRegressionCV(cv=cv, penalty='l2', solver='liblinear'))
logd1 = make_pipeline(transformer, StandardScaler(), LogisticRegressionCV(cv=cv, penalty='l2', solver='liblinear'))
logd2 = make_pipeline(transformer, StandardScaler(), LogisticRegressionCV(cv=cv, penalty='l2', solver='liblinear'))
result_log = dml_two_treatments(X, D1,D2, y, logy, logd1, logd2, nfolds=5, classifier_y = True, classifier_d1=True, classifier_d2=True,cluster = False )

In [ ]:
table_log = summary_two_treatments(*result_log, X, D1,D2, y, name1="CVE_treated", name2="OPP_treated")
table_log

### Forêts

In [ ]:
rfy = make_pipeline(transformer, RandomForestClassifier(n_estimators=100, min_samples_leaf=10, ccp_alpha=.001))
rfd1 = make_pipeline(transformer, RandomForestClassifier(n_estimators=100, min_samples_leaf=10, ccp_alpha=.001))
rfd2 = make_pipeline(transformer, RandomForestClassifier(n_estimators=100, min_samples_leaf=10, ccp_alpha=.001))
result_rf = dml_two_treatments(X, D1,D2, y, rfy, rfd1, rfd2, nfolds=5, classifier_y = True,  classifier_d1=True, classifier_d2=True,cluster = False )

In [ ]:
table_rf = summary_two_treatments(_result_rf, X, D1,D2, y, name1="CVE_treated", name2="OPP_treated")
table_rf

### Decision Tree

In [ ]:
dtry = make_pipeline(transformer, DecisionTreeClassifier(min_samples_leaf=10, ccp_alpha=.001))
dtrd1 = make_pipeline(transformer, DecisionTreeClassifier(min_samples_leaf=10, ccp_alpha=.001))
dtrd2 = make_pipeline(transformer, DecisionTreeClassifier(min_samples_leaf=10, ccp_alpha=.001))
result_dtr = dml_two_treatments(X, D1,D2, y, dtry, dtrd1, dtrd2, nfolds=5, classifier_y = True, classifier_d1=True, classifier_d2=True,cluster = False )

In [ ]:
table_dtr = summary_two_treatments(result_dtr, X, D1,D2, y, name1="CVE_treated", name2="OPP_treated")
table_dtr

### Boosted tree

In [ ]:
gbfy = make_pipeline(transformer, GradientBoostingClassifier(max_depth=2, n_iter_no_change=5))
gbfd1 = make_pipeline(transformer, GradientBoostingClassifier(max_depth=2, n_iter_no_change=5))
gbfd2 = make_pipeline(transformer, GradientBoostingClassifier(max_depth=2, n_iter_no_change=5))
result_gbf = dml_two_treatments(X, D1,D2, y, gbfy, gbfd1, gbfd2 , nfolds=5, classifier_y = True, classifier_d1=True, classifier_d2=True,cluster = False )

In [ ]:
table_gbf = summary_two_treatments(result_gbf, X, D1,D2, y, name1="CVE_treated", name2="OPP_treated")
table_gbf

### Neural networks

In [ ]:
modely = make_pipeline(StandardScaler(),
                       MLPRegressor(hidden_layer_sizes=(50, 50, 50, 50),
                                    activation='relu',
                                    solver='adam',
                                    alpha=0.0001,
                                    batch_size=200,
                                    learning_rate='constant',
                                    learning_rate_init=0.001,
                                    max_iter=200,
                                    shuffle=True,
                                    random_state=None,
                                    tol=1e-4,
                                    verbose=False,
                                    warm_start=False,
                                    momentum=0.9,
                                    nesterovs_momentum=True,
                                    early_stopping=True,
                                    validation_fraction=0.2,
                                    beta_1=0.9,
                                    beta_2=0.999,
                                    epsilon=1e-08,
                                    n_iter_no_change=10)
                       )
modeld1 = make_pipeline(StandardScaler(),
                       MLPRegressor(hidden_layer_sizes=(50, 50, 50, 50),
                                    activation='relu',
                                    solver='adam',
                                    alpha=0.0001,
                                    batch_size=200,
                                    learning_rate='constant',
                                    learning_rate_init=0.001,
                                    max_iter=200,
                                    shuffle=True,
                                    random_state=None,
                                    tol=1e-4,
                                    verbose=False,
                                    warm_start=False,
                                    momentum=0.9,
                                    nesterovs_momentum=True,
                                    early_stopping=True,
                                    validation_fraction=0.2,
                                    beta_1=0.9,
                                    beta_2=0.999,
                                    epsilon=1e-08,
                                    n_iter_no_change=10)
                       )

# Run DML model with nfolds folds of cross-fitting
result_NN = dml_two_treatments(X, D1,D2, y, modely, modeld1, modeld1 , nfolds=5,cluster = False )

In [ ]:
table_NN = summary_two_treatments(*result_NN, X, D1,D2, y, name1="CVE_treated", name2="OPP_treated")
table_NN